# (노트) Backpropagation

- toc:true
- branch: master
- badges: true
- comments: true
- author: 신록예찬
- hide: false
- categories: [빅데이터분석]

### import 

In [358]:
import torch 

### 지난시간복습

`-` 회귀분석에서 손실함수에 대한 미분은 아래와 같은 과정으로 계산할 수 있다. 

- $loss=({\bf y}-{\bf X}{\bf W})^\top({\bf y}-{\bf X}{\bf W})={\bf y}^\top {\bf y} - {\bf y}^\top {\bf X}{\bf W} - {\bf W}^\top {\bf X}^\top {\bf y} + {\bf W}^\top {\bf X}^\top{\bf X}{\bf W} $

- $\frac{\partial}{\partial {\bf W}}loss=-2 {\bf X}^\top {\bf y} + 2{\bf X}^\top{\bf X}{\bf W} $

### 오늘수업의 목표

`-` 목표: 미분을 계산할 수 있는 2개의 컴퓨터(core)가 있다고 가정할 때, 위의 계산을 더 빠르게 할 방법이 있을까? 

- 체인룰 $\to$ 역전파기법 

`-` 어떻게 하면 미분계산을 여러 컴퓨터에게 적당히 나누어 수행시킬 수 있을까? 

### 체인룰 (어려운 하나의 미분을 손쉬운 여러개의 미분으로 나누는 기법)

`-` 손실함수는 아래와 같은 변환들을 거쳐서 계산되었다고 볼 수 있다. 

- ${\bf X} \to {\bf X}{\bf W} \to {\bf y}-{\bf X}{\bf W} \to ({\bf y}-{\bf X}{\bf W})^\top({\bf y}-{\bf X}{\bf W}) $

`-` 이 과정을 좀 더 세분화 하여 나타내면 아래와 같다. 

- $\bf u = X\color{red}{W}$, $\quad {\bf u}: n \times 1 $ 

- $\bf v = y - \color{red}{u}$, $\quad {\bf v}: n \times 1 $   

- $loss= \bf \color{red}{v}^\top \color{red}{v} $  

`-` 손실함수에 대한 미분은 아래와 같다. 

$$\frac{\partial}{\partial \bf W}loss = \frac{\partial}{\partial \bf W} \bf v^\top v$$

(그런데 이걸 어떻게 계산함?)

`-` 계산할 수 있는것들의 모음.. 

- $\frac{\partial }{\partial \bf v} loss = 2{\bf v}$  $\quad \to$ (n,1) 벡터

- $\frac{\partial}{\partial \bf u} {\bf v}^\top = - {\bf I}$ $\quad \to$ (n,n) 매트릭스 

- $\frac{\partial}{\partial \bf W} {\bf u}^\top = {\bf X}^\top$ $\quad \to$ (p,n) 매트릭스 

`-` 혹시.. 아래와 같이 쓸 수 있을까? 

$$\left(\frac{\partial }{\partial \bf W}{\bf u}^\top\right) 
\left(\frac{\partial }{\partial \bf u}{\bf v}^\top \right)
\left(\frac{\partial }{\partial \bf v}loss\right) 
=\frac{\partial {\bf u}^\top}{\partial \bf W}\frac{\partial {\bf v}^\top}{\partial \bf u} \frac{\partial loss}{\partial \bf v}$$ 

- 가능할것 같다. 뭐 기호야 정의하기 나름이니까! 

`-` 그렇다면 혹시 아래와 같이 쓸 수 있을까? 

$$\frac{\partial {\bf u}^\top}{\partial \bf W}\frac{\partial {\bf v}^\top}{\partial \bf u} \frac{\partial loss}{\partial \bf v}=\frac{\partial loss}{\partial \bf W}=\frac{\partial}{\partial \bf W} loss $$

- 이건 선을 넘는 것임. 

- 그런데 어떠한 공식에 의해서 가능함. 그 공식 이름이 체인룰이다. 

`-` 결국 정리하면 아래의 꼴이 되었다. 

$$\left(\frac{\partial }{\partial \bf W}{\bf u}^\top\right) 
\left(\frac{\partial }{\partial \bf u}{\bf v}^\top \right)
\left(\frac{\partial }{\partial \bf v}loss\right) =\frac{\partial}{\partial \bf W} loss $$

`-` 그렇다면? 

$$\left({\bf X}^\top\right)
\left(-{\bf I} \right) 
\left(2{\bf v}\right) 
=\frac{\partial}{\partial \bf W} loss $$

그런데 ${\bf v}= {\bf y}-{\bf u} = {\bf y}-{\bf X}{\bf W}$ 이므로 

$$ -2{\bf X}^\top ({\bf y} - {\bf X}{\bf W})= \frac{\partial}{\partial \bf W}loss$$ 

정리하면 

$$\frac{\partial}{\partial \bf W}loss= -2{\bf X}^\top{\bf y} + 2{\bf X}^\top{\bf X}{\bf W}$$ 

### 예시: 중간고사 문제..  

`-` 미분계수를 계산하는 문제였음 

`-` 체인룰을 이용하여 미분계수를 계산하여 보자. (미분 공식을 알고 있다는 전제)

In [359]:
ones= torch.ones(5)
x = torch.tensor([11.0,12.0,13.0,14.0,15.0])
X = torch.vstack([ones,x]).T
y = torch.tensor([17.7,18.5,21.2,23.6,24.2])

In [360]:
W = torch.tensor([3.0,3.0])

In [361]:
u = X@W 
v = y-u
loss = v.T @ v

`-` $\frac{\partial}{\partial \bf v} loss$ 의 계산 

In [362]:
X.T @ -torch.eye(5) @ (2*v) 

tensor([ 209.6000, 2748.6001])

- 참고로 중간고사 답은 

```python
X.T @ -torch.eye(5) @ (2*v) 
```

이므로 `tensor([ 41.9200, 549.7200])` 와 같이 나오면 되었습니다. 

#### 방법2: 미분공식을 모를 경우.

In [363]:
u,v,loss

(tensor([36., 39., 42., 45., 48.]),
 tensor([-18.3000, -20.5000, -20.8000, -21.4000, -23.8000]),
 tensor(2212.1799))

`-` $\frac{\partial}{\partial \bf v} loss$ 의 계산

In [364]:
_v = torch.tensor([-18.3000, -20.5000, -20.8000, -21.4000, -23.8000],requires_grad=True)

In [365]:
_loss = _v.T @ _v

In [366]:
_loss.backward()

In [367]:
_v.grad.data,_v.data

(tensor([-36.6000, -41.0000, -41.6000, -42.8000, -47.6000]),
 tensor([-18.3000, -20.5000, -20.8000, -21.4000, -23.8000]))

- 미분결과가 이론적인 값인 2v와 일치함 

`-` $ \frac{\partial }{\partial \bf u} {\bf v}^\top $ 의 계산 

In [368]:
_u = torch.tensor([36., 39., 42., 45., 48.],requires_grad=True)

In [369]:
_u

tensor([36., 39., 42., 45., 48.], requires_grad=True)

In [370]:
_v= y-_u ## 아까의 _v 와 또 다른 임시 v

In [371]:
_v.T.backward() 

RuntimeError: grad can be implicitly created only for scalar outputs

- 사실 토치에서 미분계산은 스칼라만 가능

그런데 $\frac{\partial }{\partial \bf u}{\bf v}^\top=\begin{bmatrix} \frac{\partial}{\partial \bf u} v_1 \\ \frac{\partial}{\partial \bf u} v_2 \\ ... \\  \frac{\partial}{\partial \bf u} v_n \end{bmatrix} $ 이므로 아래와 같이 계산할 수 있다. 이를 이용하면 

In [372]:
u,v

(tensor([36., 39., 42., 45., 48.]),
 tensor([-18.3000, -20.5000, -20.8000, -21.4000, -23.8000]))

In [373]:
u0,u1,u2,u3,u4 = u

In [374]:
_u0=torch.tensor(36.,requires_grad=True)
_u1=torch.tensor(39.,requires_grad=True)
_u2=torch.tensor(42.,requires_grad=True)
_u3=torch.tensor(45.,requires_grad=True)
_u4=torch.tensor(48.,requires_grad=True)

In [375]:
_v0,_v1,_v2,_v3,_v4 = _u0-y[0],_u1-y[1],_u2-y[2],_u3-y[3],_u4-y[4]

In [376]:
_v0.backward()
_v1.backward()
_v2.backward()
_v3.backward()
_v4.backward()

In [378]:
_u0.grad.data

tensor(1.)

In [326]:
u_grad_data

[None, None, None, None, None]

- 미분결과가 이론적인 값인 2v와 맞는지 체크하여보자. 

In [242]:
_v[0].backward()

In [245]:
_v[1].backward()

In [246]:
u.grad.data

tensor([-1., -1., -0., -0., -0.])

In [ ]:
losses=[]

In [106]:
loss.backward()

In [107]:
W.grad.data

tensor([ 41.9200, 549.7200])

In [32]:
W = torch.tensor([2.0],requires_grad=True)

In [33]:
y-X@W 

RuntimeError: size mismatch, got 1, 1x2,1